# COT Reformatter — Gemini → Nemotron Format

Converts verbose Gemini-style numbered-step COT into:
```
<think>
natural flowing reasoning...
</think>
\boxed{final_answer}
```

**GPU target:** RTX 3060 12GB  
**Model:** Qwen2.5-3B-Instruct bf16 (~7.5 GB VRAM) — or Qwen2.5-7B-Instruct 4-bit (~5 GB VRAM)

In [ ]:
import os
print(os.environ.get("HF_HOME"))
print(os.environ.get("PIP_CACHE_DIR"))

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
INPUT_CSV   = r"generated_cot/gemini-3.1-flash-lite-preview_enhanced_sft_dataset.csv"
OUTPUT_CSV  = r"generated_cot/gemini_reformatted_sft.csv"
CKPT_CSV    = r"generated_cot/gemini_reformatted_sft_ckpt.csv"  # resume checkpoint

# Model selection:
#   Option A (recommended for 3060 12 GB): 3B bf16 — fast, ~7.5 GB VRAM, batch 4
#   Option B (higher quality):            7B 4-bit — ~5 GB VRAM, batch 2
MODEL_ID     = "Qwen/Qwen2.5-3B-Instruct"
LOAD_IN_4BIT = False   # set True + MODEL_ID = "Qwen/Qwen2.5-7B-Instruct" for option B

BATCH_SIZE       = 4     # safe for 3B bf16 on 12 GB; drop to 2 if OOM
MAX_INPUT_TOKENS = 8192  # truncate long COTs in prompt (keeps last N tokens)
MAX_NEW_TOKENS   = 8192  # max tokens the model generates
SAVE_EVERY       = 5   # checkpoint frequency (rows)

DEVICE = "cuda"
# ──────────────────────────────────────────────────────────────────────────

In [ ]:
# Install if needed (comment out if already installed)
# !pip install -q transformers accelerate bitsandbytes tqdm pandas

In [ ]:
import torch, pandas as pd, re, json
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────
quant_cfg = None
if LOAD_IN_4BIT:
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"  # decoder-only: pad left for batched generation

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_cfg,
    torch_dtype=torch.bfloat16 if not LOAD_IN_4BIT else None,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Model loaded | VRAM used: {used:.1f} / {total:.1f} GB")

In [ ]:
# ── Prompt template ───────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a reasoning trace reformatter. \
Convert the provided numbered step-by-step analysis into a natural, \
flowing chain-of-thought. Remove verbose headers like '1. **Analyze the Request:**'. \
Write as if thinking through the problem yourself. \
Wrap the reasoning in <think>...</think> tags. \
End with the final answer on its own line as \\boxed{answer}. \
Do NOT change the final answer value."""

# Few-shot example (bit manipulation)
FEW_SHOT_USER = """Original reasoning:
1. **Analyze the Request:** Determine the 8-bit transformation for input 00110100.
2. **Examine the Data:** Input 01010001 -> Output 11011101; the pattern appears to involve bit reversal and XOR.
3. **Apply Rule:** Reverse bits of 00110100 = 00101100, XOR with 11111111 = 11010011... re-checking each example reveals it is a permutation plus constant XOR.
4. **Final Output Formulation:** The result is 10010111.

Final answer: 10010111"""

FEW_SHOT_ASSISTANT = """<think>
Looking at the examples, I need to find the transformation that maps each 8-bit input to its output. Let me examine the pairs: 01010001 -> 11011101, 00001001 -> 01101101, etc.

The pattern appears to involve a combination of bit permutation and XOR with a constant. Testing bit reversal on 01010001 gives 10001010, which doesn't match 11011101 directly. Let me check if there's a positional swap pattern.

After checking multiple examples, the rule maps each bit position through a specific permutation, then XORs with a constant. Applying this derived rule to 00110100 yields 10010111.
</think>
\\boxed{10010111}"""

def build_messages(cot: str, answer: str) -> list[dict]:
    """Build chat messages for the reformatter."""
    user_content = (
        f"Original reasoning:\n{cot}\n\nFinal answer: {answer}"
    )
    return [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": FEW_SHOT_USER},
        {"role": "assistant", "content": FEW_SHOT_ASSISTANT},
        {"role": "user",      "content": user_content},
    ]

In [ ]:
# ── Validation helpers ────────────────────────────────────────────────────
def extract_last_boxed(text: str) -> str | None:
    """Brace-balanced extraction of last \\boxed{}."""
    last_start = None
    i = 0
    while i < len(text):
        idx = text.find(r'\boxed{', i)
        if idx == -1:
            break
        last_start = idx + len(r'\boxed{')
        i = last_start
    if last_start is None:
        return None
    depth, j = 1, last_start
    while j < len(text) and depth > 0:
        if text[j] == '{': depth += 1
        elif text[j] == '}': depth -= 1
        j += 1
    return text[last_start:j-1].strip() if depth == 0 else None

def is_valid_output(text: str, expected_answer: str) -> bool:
    """Check <think>...</think> present and boxed answer matches."""
    has_think_open  = '<think>' in text
    has_think_close = '</think>' in text
    boxed = extract_last_boxed(text)
    if boxed is None:
        return False
    norm = lambda s: ' '.join(str(s).strip().lower().split())
    answer_ok = norm(boxed) == norm(expected_answer)
    return has_think_open and has_think_close and answer_ok

# Quick test
test_out = '<think>reasoning here</think>\n\\boxed{10010111}'
assert is_valid_output(test_out, '10010111'), "validator broken"
print("Validator OK")

In [ ]:
# ── Batch reformat function ───────────────────────────────────────────────
@torch.inference_mode()
def reformat_batch(rows: list[dict]) -> list[str]:
    """
    rows: list of {cot, answer}
    Returns list of raw model output strings.
    """
    # Build prompt strings via chat template
    prompts = []
    for row in rows:
        msgs = build_messages(row["cot"], row["answer"])
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(text)

    # Tokenize with truncation on the input side
    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS + 256,  # +256 for template overhead
    ).to(DEVICE)

    input_len = enc["input_ids"].shape[1]

    out = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,          # greedy — reformatting is deterministic
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Decode only generated tokens
    decoded = tokenizer.batch_decode(
        out[:, input_len:],
        skip_special_tokens=True,
    )
    return decoded

In [ ]:
# ── Sanity check — single row before full run ─────────────────────────────
df_raw = pd.read_csv(INPUT_CSV, encoding="utf-8")
sample = df_raw.iloc[2]  # cipher row: 'cat imagines book'

test_out = reformat_batch([{"cot": sample["generated_cot"], "answer": sample["answer"]}])[0]
print("=== MODEL OUTPUT ===")
print(test_out)
print("\n=== VALID:", is_valid_output(test_out, sample["answer"]))

In [ ]:
# ── Main processing loop ──────────────────────────────────────────────────
df_raw = pd.read_csv(INPUT_CSV, encoding="utf-8")
ckpt_path = Path(CKPT_CSV)

# Resume from checkpoint
if ckpt_path.exists():
    df_done = pd.read_csv(ckpt_path, encoding="utf-8")
    done_ids = set(df_done["id"].astype(str))
    df_todo  = df_raw[~df_raw["id"].astype(str).isin(done_ids)].reset_index(drop=True)
    results  = df_done.to_dict("records")
    print(f"Resuming: {len(done_ids)} done, {len(df_todo)} remaining")
else:
    df_todo = df_raw.copy().reset_index(drop=True)
    results = []
    print(f"Starting fresh: {len(df_todo)} rows")

n_valid   = sum(1 for r in results if r.get("_reformat_valid", False))
n_invalid = sum(1 for r in results if not r.get("_reformat_valid", False))

rows_list = df_todo.to_dict("records")

for batch_start in tqdm(range(0, len(rows_list), BATCH_SIZE), desc="Reformatting"):
    batch = rows_list[batch_start : batch_start + BATCH_SIZE]
    inputs = [{"cot": r["generated_cot"], "answer": r["answer"]} for r in batch]

    try:
        outputs = reformat_batch(inputs)
    except torch.cuda.OutOfMemoryError:
        print(f"OOM at batch {batch_start} — reduce BATCH_SIZE and retry")
        torch.cuda.empty_cache()
        # Fallback: process one by one
        outputs = []
        for inp in inputs:
            try:
                o = reformat_batch([inp])[0]
            except torch.cuda.OutOfMemoryError:
                o = ""
                torch.cuda.empty_cache()
            outputs.append(o)

    for row, out in zip(batch, outputs):
        valid = is_valid_output(out, row["answer"])
        n_valid   += valid
        n_invalid += not valid
        results.append({
            "id":              row["id"],
            "prompt":          row["prompt"],
            "answer":          row["answer"],
            "reformatted_cot": out.strip(),
            "_reformat_valid": valid,
        })

    # Checkpoint
    if (batch_start // BATCH_SIZE + 1) % (SAVE_EVERY // BATCH_SIZE) == 0:
        pd.DataFrame(results).to_csv(ckpt_path, index=False, encoding="utf-8")

print(f"\nDone. Valid: {n_valid}/{len(results)} ({n_valid/len(results)*100:.1f}%)  Invalid: {n_invalid}")

In [ ]:
# ── Save final CSV — valid rows only ─────────────────────────────────────
df_out = pd.DataFrame(results)

# Save ALL (including invalid, flagged) for inspection
df_out.to_csv(ckpt_path, index=False, encoding="utf-8")

# Save valid-only for training
df_valid = (
    df_out[df_out["_reformat_valid"]]
    .drop(columns=["_reformat_valid"])
    .reset_index(drop=True)
)
df_valid.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"Saved {len(df_valid):,} valid rows → {OUTPUT_CSV}")
print(f"Saved {len(df_out):,} all rows   → {CKPT_CSV}")

# Sample
sample_row = df_valid.iloc[0]
print("\n=== SAMPLE OUTPUT ===")
print("answer:", sample_row["answer"])
print(sample_row["reformatted_cot"])

In [ ]:
# ── Inspect invalid rows ──────────────────────────────────────────────────
df_invalid = df_out[~df_out["_reformat_valid"]].reset_index(drop=True)
print(f"Invalid rows: {len(df_invalid)}")

if len(df_invalid):
    for i, row in df_invalid.head(5).iterrows():
        print(f"\n--- Row {i} | answer={row['answer']!r} ---")
        print(repr(row["reformatted_cot"][-300:]))

    # Diagnose failure modes
    no_think     = df_invalid["reformatted_cot"].apply(lambda x: '<think>' not in str(x)).sum()
    no_boxed     = df_invalid["reformatted_cot"].apply(lambda x: r'\boxed{' not in str(x)).sum()
    wrong_answer = len(df_invalid) - no_think - no_boxed  # rough

    print(f"\nMissing <think> : {no_think}")
    print(f"Missing \\boxed  : {no_boxed}")
    print(f"Wrong answer    : {wrong_answer} (approx)")

In [ ]:
# ── VRAM summary ──────────────────────────────────────────────────────────
if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1e9
    curr = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Peak VRAM : {peak:.2f} GB")
    print(f"Curr VRAM : {curr:.2f} GB")
    print(f"Total     : {total:.2f} GB")